# E6 | Model Risk XGBoost + SHAP
Classificar incidentes em risco de violar OLA com explicabilidade SHAP

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import shap
import mlflow

load_dotenv()
print('Setup OK')

In [ ]:
# RDS Connection
RDS_HOST = os.getenv('RDS_HOST')
RDS_USER = os.getenv('RDS_USER')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')
RDS_DATABASE = os.getenv('RDS_DATABASE')

engine = create_engine(f'postgresql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:5432/{RDS_DATABASE}')
print('RDS Connected')

In [ ]:
# Ler SLA classification dataset
df = pd.read_sql('SELECT * FROM gold.ml_sla_classification_dataset', engine)
print(f'Loaded {len(df)} records')
print(f'Target distribution: {df.target_risco_sla.value_counts().to_dict()}')

In [ ]:
# Preparar features e target
feature_cols = [c for c in df.columns if c not in ['incident_id', 'target_risco_sla', 'data_abertura']]
X = df[feature_cols].fillna(0)
y = df['target_risco_sla']

print(f'Features: {len(feature_cols)}')
print(f'Target class balance: {y.value_counts().to_dict()}')

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# Treinar XGBoost
model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
print('Training XGBoost...')
model.fit(X_train, y_train, verbose=0)
print('Done')

In [ ]:
# Avaliar
y_pred_proba = model.predict_proba(X_test)[:, 1]
auc_roc = roc_auc_score(y_test, y_pred_proba)

y_pred = model.predict(X_test)

print(f'AUC-ROC: {auc_roc:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# SHAP explicabilidade
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print(f'SHAP values shape: {shap_values.shape}')
print(f'Top features by SHAP importance:')

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'shap_importance': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

print(feature_importance.head(10))

In [ ]:
# MLflow
mlflow.set_experiment('xgboost_ola_risk')
with mlflow.start_run():
    mlflow.log_params({
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.1
    })
    mlflow.log_metrics({
        'auc_roc': auc_roc,
        'n_features': len(feature_cols)
    })
    mlflow.sklearn.log_model(model, 'xgboost_model')
    print('Logged to MLflow')